# F1 Race Strategy Engine — How It Works

This notebook teaches all 5 layers of the engine step by step.

**Layers:**
1. Data collection (FastF1)
2. ML model (RandomForest)
3. Race simulation (lap-by-lap loop)
4. Strategy optimizer (brute-force search)
5. Monte Carlo (uncertainty modeling)

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0e1117'
matplotlib.rcParams['axes.facecolor'] = '#1c1f26'
matplotlib.rcParams['axes.edgecolor'] = '#374151'
matplotlib.rcParams['text.color'] = '#f9fafb'
matplotlib.rcParams['axes.labelcolor'] = '#9ca3af'
matplotlib.rcParams['xtick.color'] = '#9ca3af'
matplotlib.rcParams['ytick.color'] = '#9ca3af'
print('Libraries loaded.')

## Layer 1 — Data Collection

FastF1 gives us real lap telemetry. Each row = one lap.

In [ ]:
# Try real data first, fall back to synthetic
import os
if os.path.exists('../data/lap_data.csv'):
    df = pd.read_csv('../data/lap_data.csv')
    print(f'Loaded {len(df)} real laps from FastF1')
else:
    from train_model import generate_synthetic_data
    df = generate_synthetic_data(3000)
    print(f'Generated {len(df)} synthetic laps')
df.head(10)

In [ ]:
# Visualise the raw data
fig, ax = plt.subplots(figsize=(12, 4))
colors = {'SOFT': '#e10600', 'MEDIUM': '#fbbf24', 'HARD': '#9ca3af'}
for compound, color in colors.items():
    subset = df[df['Compound'] == compound]
    ax.scatter(subset['TyreLife'], subset['LapTimeSeconds'],
               alpha=0.3, s=8, color=color, label=compound)
ax.set_xlabel('Tyre Life (laps)')
ax.set_ylabel('Lap Time (s)')
ax.set_title('Raw lap data — tyre life vs lap time by compound')
ax.legend()
plt.tight_layout()
plt.show()
print('Notice how SOFT degrades fastest — the spread widens at higher tyre life')

## Layer 2 — ML Model

RandomForest learns the degradation curve from this data.

In [ ]:
# Train (or load if already trained)
import sys; sys.path.insert(0, '..')
from train_model import train_model
model, encoder = train_model()
print('Model ready.')

In [ ]:
# Visualise what the model learned — degradation curves
from backend.simulator import predict_lap_time

tyre_lives = np.arange(1, 50)
fig, ax = plt.subplots(figsize=(12, 5))
for compound, color in colors.items():
    preds = [predict_lap_time(model, encoder, compound, tl, 30) for tl in tyre_lives]
    ax.plot(tyre_lives, preds, color=color, label=compound, linewidth=2)
ax.set_xlabel('Tyre life (laps)')
ax.set_ylabel('Predicted lap time (s)')
ax.set_title('ML model output — predicted degradation curves')
ax.legend()
plt.tight_layout()
plt.show()
print('The model correctly learned: SOFT degrades fastest, HARD most consistent')

## Layer 3 — Race Simulation

The simulator runs the ML model 60 times, one per lap.

In [ ]:
from backend.simulator import simulate_race

# Simulate a race with a pit stop on lap 26
result = simulate_race(model, encoder,
    start_compound='MEDIUM',
    pit_laps=[26],
    pit_stop_loss=22.0,
    total_laps=60
)

print(f'Total race time: {result["total_time"]:.1f}s')
print(f'Pit events: {result["events"]}')

# Plot lap-by-lap
fig, ax = plt.subplots(figsize=(14, 4))
bar_colors = [colors[c] for c in result['compounds']]
ax.bar(range(1, 61), result['lap_times'], color=bar_colors)
ax.axvline(x=26, color='#3b82f6', linestyle='--', label='Pit stop')
ax.set_xlabel('Lap number')
ax.set_ylabel('Lap time (s)')
ax.set_title('simulate_race() output — lap-by-lap times (colour = compound)')
ax.legend()
plt.tight_layout()
plt.show()

## Layer 4 — Strategy Optimizer

Brute-force: try every possible pit lap, pick the minimum.

In [ ]:
from backend.simulator import find_best_one_stop, find_best_two_stop

r1 = find_best_one_stop(model, encoder, 'MEDIUM', 22.0, 60)
r2 = find_best_two_stop(model, encoder, 'MEDIUM', 22.0, 60)

print(f'Best 1-stop: lap {r1["best_lap"]} → {r1["best_time"]:.1f}s')
print(f'Best 2-stop: laps {r2["best_laps"]} → {r2["best_time"]:.1f}s')
print(f'Recommended: {"2-stop" if r2["best_time"] < r1["best_time"] else "1-stop"}')

In [ ]:
# Plot the full optimizer search
pit_laps_range = list(r1['all_results'].keys())
times = list(r1['all_results'].values())

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pit_laps_range, times, color='#3b82f6', linewidth=2)
ax.axvline(x=r1['best_lap'], color='#e10600', linestyle='--',
           label=f'Optimal: lap {r1["best_lap"]}')
ax.fill_between(pit_laps_range, times, alpha=0.1, color='#3b82f6')
ax.set_xlabel('Pit lap')
ax.set_ylabel('Total race time (s)')
ax.set_title('Optimizer search — race time vs every possible pit lap')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Spread: {max(times)-min(times):.1f}s between best and worst pit lap')

## Layer 5 — Monte Carlo

Run 500 simulations with noise. Gives a distribution, not just one number.

In [ ]:
from backend.simulator import monte_carlo_simulation

mc = monte_carlo_simulation(
    model, encoder, 'MEDIUM',
    pit_laps=[r1['best_lap']],
    pit_stop_loss=22.0,
    total_laps=60,
    n_simulations=500,
    noise_std=0.4,
)

print(f'Mean:       {mc["mean"]:.1f}s')
print(f'Std dev:    {mc["std"]:.2f}s')
print(f'P5  (best): {mc["p5"]:.1f}s')
print(f'P95 (worst):{mc["p95"]:.1f}s')
print(f'90% CI:     {mc["p95"]-mc["p5"]:.1f}s')

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(mc['all_times'], bins=40, color='#3b82f6', alpha=0.7, edgecolor='none')
ax.axvline(x=mc['mean'], color='#e10600', linestyle='--', label=f'Mean: {mc["mean"]:.1f}s')
ax.axvspan(mc['p5'], mc['p95'], alpha=0.15, color='#3b82f6', label='90% CI')
ax.set_xlabel('Total race time (s)')
ax.set_ylabel('Frequency')
ax.set_title('Monte Carlo distribution — 500 simulated race outcomes')
ax.legend()
plt.tight_layout()
plt.show()

## Bonus — Full 2026 Driver Comparison

In [ ]:
from backend.simulator import driver_comparison

results = driver_comparison(model, encoder, race_name='Monaco',
                             start_compound='MEDIUM', pit_stop_loss=22.0)

df_res = pd.DataFrame([{
    'Pos': r['position'], 'Driver': r['driver'], 'Team': r['team'],
    'Strategy': r['strategy'], 'Pit laps': str(r['pit_laps']),
    'Race time': f"{int(r['race_time']//60)}:{r['race_time']%60:.2f}",
    'Gap': f"+{r['gap']:.1f}s"
} for r in results])
print(df_res.to_string(index=False))